In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/tw_rp_main/datasets/NAFinal.csv"  # queries
B_PATH = "/home/ubuntu/tw_rp_main/datasets/500_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_3_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_3_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/tw_rp_main/similarity_scores/cross_similar_posts_k3_100_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['body'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(                                           _text_key  \
 0  8w1d - just found out i'm having twins 😲 || my...   
 1  imperforate hymen || hi, i’m 18 years old fema...   
 2  ovulation after d&c || has anyone ovulated as ...   
 3  what is the likelihood of an abnormal colposco...   
 4  i am "that" awful ex who everyone dreads...aft...   
 
                                                title  \
 0           8w1d - just found out I'm having twins 😲   
 1                                  IMPERFORATE HYMEN   
 2                                Ovulation after D&C   
 3  What is the likelihood of an abnormal colposco...   
 4  I am "that" awful ex who everyone dreads...aft...   
 
                                                 body Unnamed: 3  \
 0  My husband and I are in total shock!  We went ...        NaN   
 1  hi, i’m 18 years old female and i live in Cana...        NaN   
 2  Has anyone ovulated as soon as 6 days post D&C...        NaN   
 3  28F. Been dealing with spotting and 

In [7]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r5_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r5_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r5_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r5_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [8]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [9]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r5_emb_A, r5_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/tw_rp_main/similarity_scores/cross_similar_posts_k3_60_test.json
